# Reeling in Smart Attackers' Phishing Emails — Colab notebook

**CSE 587 — Group 5**: Parker Davis · Dillon VanGilder · Emmanuel Adjei Domfeh.

End-to-end pipeline:
1. Install dependencies
2. Get the project source (clone / upload / mount Drive)
3. Download the Champa et al. curated phishing dataset
4. Train BERT baseline
5. Evaluate on standard test + smart-attacker test
6. Train **with smart-attacker augmentation**
7. Compare the three stages

Runtime: GPU (Runtime → Change runtime type → T4 GPU).

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.45.2 datasets==2.21.0 accelerate==0.34.2 \
    scikit-learn pandas matplotlib seaborn nltk requests pyarrow

## 2. Get the project source

Pick **one** of the cells below.

**Option A — Mount Google Drive** (recommended if you've copied the project folder there):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/cse587

**Option B — Upload the project as a zip from your laptop:**

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # pick cse587.zip
# !unzip -q cse587.zip -d /content/
# %cd /content/cse587

**Option C — Clone from GitHub (after you push the project):**

In [ ]:
# !git clone https://github.com/<your-username>/cse587-phishing.git
# %cd cse587-phishing

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
print('CWD:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

## 3. Download the dataset (Champa et al. 2024)

In [ ]:
!python scripts/download_data.py
!ls -lh data/raw

## 4. Inspect the dataset

In [ ]:
from src.data_loader import load_raw_csvs
df = load_raw_csvs()
print('Total rows:', len(df))
print(df.label.value_counts())
df.head(3)

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
sns.countplot(x='label', data=df)
plt.title('Class balance (0 = benign, 1 = phishing)')
plt.show()

## 5. Train baseline BERT

In [ ]:
!python -m src.train --model bert-base-uncased --epochs 3 --batch_size 16

## 6. Train with smart-attacker augmentation

In [ ]:
!python -m src.train --model bert-base-uncased --epochs 3 --augment --augment_factor 1

## 7. Compare reports

In [ ]:
import json, pandas as pd
from pathlib import Path

def load_report(p):
    return json.loads(Path(p).read_text())

rb = load_report('results/baseline/../eval_report.json') if Path('results/eval_report.json').exists() else None
ra = load_report('results/eval_report.json')  # last run's report
ra

In [ ]:
# Visualize: standard vs smart-attacker for the augmented run
import matplotlib.pyplot as plt
labels = ['accuracy', 'f1_phish', 'recall_phish', 'macro_f1']
std = [ra['standard'][k] for k in labels]
sm  = [ra['smart_attacker'][k] for k in labels]
x = range(len(labels))
fig, ax = plt.subplots(figsize=(7, 4))
w = 0.35
ax.bar([i - w/2 for i in x], std, w, label='Standard test')
ax.bar([i + w/2 for i in x], sm,  w, label='Smart-attacker test')
ax.set_xticks(list(x)); ax.set_xticklabels(labels, rotation=20)
ax.set_ylim(0, 1)
ax.legend()
ax.set_title('Standard vs smart-attacker test (last run)')
plt.tight_layout(); plt.show()

## 8. Per-edit ablation

How much does each individual smart-attacker edit hurt the model?

In [ ]:
abl = ra['ablation']
ops = list(abl.keys())
f1s = [abl[o]['f1_phish'] for o in ops]
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(ops, f1s, color='steelblue')
ax.axhline(ra['standard']['f1_phish'], color='black', linestyle='--', label='standard F1')
ax.set_ylim(0, 1); ax.set_ylabel('F1 (phishing)')
ax.set_title('F1 drop when each smart-attacker edit is applied alone')
plt.xticks(rotation=20); plt.legend(); plt.tight_layout(); plt.show()

## 9. Inspect a few smart-attacker examples

In [ ]:
from src.augmentation import AugmentationConfig, make_smart_attacker_eval
from src.data_loader import load_splits
splits = load_splits()
smart = make_smart_attacker_eval(splits['test'].head(20).copy(), AugmentationConfig())
for _, row in smart[smart.label == 1].head(3).iterrows():
    print('---')
    print(row['text'][:600])